<a href="https://colab.research.google.com/github/munizgui/CP4_SERS/blob/main/CP4_SERS_EX6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset 6 — Individual Household Electric Power Consumption (UCI)

Fonte: https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

Situação

Uma residência possui monitoramento elétrico detalhado e deseja identificar episódios de demanda elevada que também apresentem corrente acima do comportamento médio. O dataset é o mesmo utilizado como referência em aula, mas o critério de análise será diferente.

In [ ]:
import pandas as pd

In [ ]:
# 1. Carregue a nova amostra e trate dados ausentes se necessário
dados = pd.read_csv("household_power_consumption.csv")

if dados.iloc[0].astype(str).str.contains('continuous|discrete|string|time|meta').any():
    dados = dados.iloc[2:].reset_index(drop=True)

# Valores ausentes ('?') e converte para numérico
dados.replace('?', pd.NA, inplace=True)
dados.dropna(inplace=True)

for col in dados.columns:
    dados[col] = pd.to_numeric(dados[col], errors='coerce')

dados.head(10)

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.256,0.106,242.00,1.2,0.0,0.0,1.0
1,0.466,0.352,237.22,2.4,0.0,2.0,0.0
2,0.758,0.194,238.66,3.2,0.0,1.0,0.0
3,1.290,0.046,240.64,5.4,1.0,0.0,18.0
4,0.428,0.202,242.23,1.8,0.0,2.0,1.0
6,1.206,0.302,242.10,5.0,0.0,1.0,12.0
7,0.182,0.120,241.84,0.8,0.0,1.0,0.0
8,1.792,0.068,242.39,7.4,0.0,0.0,18.0
9,0.474,0.328,242.28,2.2,0.0,1.0,1.0
11,0.214,0.120,240.88,1.0,0.0,2.0,0.0


In [ ]:
dados.shape

(204923, 7)

In [ ]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
Index: 204923 entries, 0 to 207525
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Global_active_power    204923 non-null  float64
 1   Global_reactive_power  204923 non-null  float64
 2   Voltage                204923 non-null  float64
 3   Global_intensity       204923 non-null  float64
 4   Sub_metering_1         204923 non-null  float64
 5   Sub_metering_2         204923 non-null  float64
 6   Sub_metering_3         204923 non-null  float64
dtypes: float64(7)
memory usage: 12.5 MB


In [ ]:
dados.describe()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
count,204923.000000,204923.000000,204923.000000,204923.000000,204923.000000,204923.000000,204923.000000
mean,1.093833,0.124067,240.843703,4.637612,1.134158,1.293671,6.486690
std,1.058912,0.113045,3.237193,4.451022,6.197910,5.830441,8.439974
min,0.078000,0.000000,224.670000,0.200000,0.000000,0.000000,0.000000
25%,0.310000,0.048000,239.000000,1.400000,0.000000,0.000000,0.000000
50%,0.604000,0.100000,241.020000,2.600000,0.000000,0.000000,1.000000
75%,1.530000,0.194000,242.890000,6.400000,0.000000,1.000000,17.000000
max,10.650000,1.390000,253.750000,46.400000,88.000000,77.000000,31.000000


In [ ]:
dados.rename(columns={
    'Global_active_power': 'Potencia_Ativa',
    'Global_reactive_power': 'Potencia_Reactiva',
    'Voltage': 'Tensao',
    'Global_intensity': 'Corrente',
    'Sub_metering_1': 'Cozinha',
    'Sub_metering_2': 'Lavanderia',
    'Sub_metering_3': 'Climatizacao'
}, inplace=True)

dados.columns

Index(['Potencia_Ativa', 'Potencia_Reactiva', 'Tensao', 'Corrente', 'Cozinha',
       'Lavanderia', 'Climatizacao'],
      dtype='object')

In [ ]:
# 2. Determine o valor máximo de potência ativa
max_potencia_ativa = dados['Potencia_Ativa'].max()

print(f"O valor máximo de potência ativa registrado foi: {max_potencia_ativa} kW")

O valor máximo de potência ativa registrado foi: 10.65 kW


In [ ]:
# 3. Calcule 75% do valor máximo e crie um DataFrame contendo registros acima desse limite
limiar_potencia = max_potencia_ativa * 0.75

df_alta_potencia = dados[dados['Potencia_Ativa'] > limiar_potencia]

print(f"Limiar de 75% da Potência Ativa: {limiar_potencia:.2f} kW")
df_alta_potencia.head()

Limiar de 75% da Potência Ativa: 7.99 kW


,Potencia_Ativa,Potencia_Reactiva,Tensao,Corrente,Cozinha,Lavanderia,Climatizacao
24635,8.110,0.196,236.87,34.2,37.0,74.0,17.0
42771,8.026,0.058,234.91,34.0,37.0,73.0,17.0
45962,8.218,0.336,236.05,34.8,37.0,73.0,17.0
46828,9.708,0.372,231.37,42.0,37.0,69.0,17.0
55667,8.168,0.540,228.91,35.6,35.0,35.0,17.0


In [ ]:
# 4. Calcule a quantidade e o percentual de registros selecionados
total_registros = len(dados)
qtd_alta_potencia = len(df_alta_potencia)
pct_alta_potencia = (qtd_alta_potencia / total_registros) * 100

print(f"Total de registros na amostra: {total_registros}")
print(f"Registros com Potência Ativa > 75%: {qtd_alta_potencia}")
print(f"Percentual da amostra: {pct_alta_potencia:.2f}%")

Total de registros na amostra: 204923
Registros com Potência Ativa > 75%: 33
Percentual da amostra: 0.02%


In [ ]:
# 5. Calcule a corrente média da amostra
corrente_media = dados['Corrente'].mean()

print(f"A corrente média (Global_intensity) da amostra é: {corrente_media:.2f} A")

A corrente média (Global_intensity) da amostra é: 4.64 A


In [ ]:
# 6. Crie um segundo DataFrame contendo simultaneamente potência ativa acima de 75% do máximo e corrente acima da média
df_potencia_e_corrente = dados[
    (dados['Potencia_Ativa'] > limiar_potencia) &
    (dados['Corrente'] > corrente_media)
]

qtd_duplo_criterio = len(df_potencia_e_corrente)
pct_duplo_criterio = (qtd_duplo_criterio / total_registros) * 100

print(f"Registros com Potência Ativa > 75% E Corrente > Média: {qtd_duplo_criterio}")
print(f"Percentual da amostra: {pct_duplo_criterio:.2f}%")

df_potencia_e_corrente.head()

Registros com Potência Ativa > 75% E Corrente > Média: 33
Percentual da amostra: 0.02%


,Potencia_Ativa,Potencia_Reactiva,Tensao,Corrente,Cozinha,Lavanderia,Climatizacao
24635,8.110,0.196,236.87,34.2,37.0,74.0,17.0
42771,8.026,0.058,234.91,34.0,37.0,73.0,17.0
45962,8.218,0.336,236.05,34.8,37.0,73.0,17.0
46828,9.708,0.372,231.37,42.0,37.0,69.0,17.0
55667,8.168,0.540,228.91,35.6,35.0,35.0,17.0


# 7. Compare os dois conjuntos e explique o efeito da inclusão da corrente como segunda condição.

A adição da corrente elétrica como segundo critério filtra a análise para focar exclusivamente nos momentos de pico de demanda ativa em que o circuito também esteve sob maior carga de corrente elétrica.

A inclusão dessa segunda condição aumenta o rigor da filtragem. Como a potência ativa e a corrente elétrica possuem uma relação física diretamente proporcional, quase a totalidade dos picos de potência ativa extrema coincide com momentos de corrente elevada. No entanto, a inclusão do critério de corrente garante a eliminação de eventuais oscilações pontuais de potência causadas por flutuações de tensão sem elevação na intensidade da corrente, permitindo isolar com precisão os episódios em que a fiação e os circuitos da residência sofreram maior estresse elétrico real.